In [1]:
#Apply Raw
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch

In [2]:
#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

,person_ID,acc_x,acc_y,acc_z,eda,bvp,hr,temp,pain_scale
0,P001,0.2751,-0.0464,0.3049,0.7395,99.24,67.6,33.94,5
1,P001,0.2428,-0.1161,0.3641,0.7793,103.24,68.3,33.95,5
2,P001,0.0146,-0.1479,0.6552,0.8581,103.08,68.1,33.91,5
3,P001,-0.0806,-0.2144,0.6631,0.8881,104.12,66.6,33.94,5
4,P001,-0.0808,-0.1754,0.5448,0.7786,107.05,66.8,33.95,5
...,...,...,...,...,...,...,...,...,...
95995,P200,0.3618,0.0199,0.1452,5.2451,124.71,104.4,36.09,7
95996,P200,0.2842,-0.1367,0.0158,5.3054,125.71,103.4,36.09,7
95997,P200,0.3005,-0.1288,-0.1729,5.2345,125.07,103.3,36.13,7
95998,P200,0.2964,-0.1015,-0.2274,5.2046,126.12,102.7,36.19,7


In [20]:
# Drop target and ID column & target column
X_raw = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape (raw version):", X_raw.shape)


#Define Cluster Parameters
k_values = range(2, 9)  # clusters 2–8 for KMeans, GMM, Agglomerative, Spectral
n_init = 10  # random initialization for KMeans, GMM, Spectral
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

Features shape (raw version): (96000, 7)


In [21]:
#K-Means
start_time = time.time()
kmeans_raw = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    kmeans.fit(X_raw)
    labels = kmeans.labels_
    sil, db, ch = compute_metrics(X_raw, labels) #K-Means on Raw Featureslabels)
    kmeans_raw.append({"algorithm":"K-Means","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 1191.051700592041 seconds
K-Means runtime: 1191.0517 seconds


In [22]:
#GMM on Raw Features
start_time = time.time()
gmm_raw = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    gmm.fit(X_raw)
    labels = gmm.predict(X_raw)
    sil, db, ch = compute_metrics(X_raw, labels)
    gmm_raw.append({"algorithm":"GMM","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 1569.2603607177734 seconds
GMM runtime: 1569.2604 seconds


In [17]:
#Agglomerative Clustering on Scaled + PCA Data
df_small = df.sample(n=5000, random_state=42)
X = df_small.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape (raw version):", X.shape)
start_time = time.time()
agg_raw = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg.fit(X)
    labels = agg.labels_
    sil, db, ch = compute_metrics(X, labels)
    agg_raw.append({"algorithm":"Agglomerative","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Features shape (raw version): (5000, 7)
Runtime: 7.032531499862671 seconds
Agglomerative runtime: 7.0325 seconds


In [23]:
#Spectral Clustering
start_time = time.time()
spectral_raw = []
for k in k_values:
    spectral = SpectralClustering(n_clusters=k, affinity='nearest_neighbors', n_init=n_init, random_state=42)
    spectral.fit(X_raw)
    labels = spectral.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    spectral_raw.append({"algorithm":"Spectral","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")

Runtime: 2623.0251157283783 seconds
Spectral runtime: 2623.0251 seconds


In [ ]:
#DBSCAN
start_time = time.time()
dbscan_raw = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    dbscan.fit(X_raw)
    labels = dbscan.labels_
    sil, db, ch = compute_metrics(X_raw, labels)
    dbscan_raw.append({"algorithm":"DBSCAN","preprocessing":"raw","eps":eps,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"DBSCAN runtime: {runtime:.4f} seconds") 

Runtime: 553.1269352436066 seconds
DBSCAN runtime: 553.1269 seconds


In [16]:
#Birch
start_time = time.time()
df_small = df.sample(n=5000, random_state=42)
X = df_small.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape (raw version):", X.shape)
birch_raw = []
threshold_values = [0.2, 0.5, 1.0, 1.5]

for t in threshold_values:
    birch = Birch(n_clusters=None, threshold=t)
    labels = birch.fit_predict(X)

    if len(set(labels)) > 1:
        sil, db, ch = compute_metrics(X, labels)
        birch_raw.append({
            "algorithm": "BIRCH",
            "preprocessing": "raw",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Birch runtime: {runtime:.4f} seconds")

Features shape (raw version): (5000, 7)
Runtime: 9.205454349517822 seconds
Birch runtime: 9.2055 seconds


In [25]:
#OPTICS
start_time = time.time()
optics_raw = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_raw)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_raw, labels)
        optics_raw.append({
            "algorithm": "OPTICS",
            "preprocessing": "raw",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 12574.187056541443 seconds
Optics runtime: 12574.1871 seconds


In [26]:
import csv

pain_results_raw = (kmeans_raw+gmm_raw+agg_raw+spectral_raw+dbscan_raw+birch_raw + optics_raw)


keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/pain_data/pain_raw.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(pain_results_raw)